<a href="https://colab.research.google.com/github/furalmiqbadi/PembelajaranMesin_01_AbdulGhofurAlmiqbadi/blob/main/JS09/JS09_Tugas02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Langkah 1 - Install Library

In [2]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 6.7 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 31.8 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520956 sha256=61694f6260bd8bc731ef8f2b34509b1722ac518da3ec3b7d8fc9026646bcc9bc
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679064 sha256=86d80275aca71b67f84c7a2423c75c2d851b6f37fb5425cc9dd21f7a431c5a49
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [6]:
import kagglehub
import os

path = kagglehub.dataset_download("bwandowando/spotify-songs-with-attributes-and-lyrics")

print("Path ke folder dataset:", path)

target_file = 'songs_with_attributes_and_lyrics.csv'
files = os.listdir(path)

if target_file in files:
    csv_path = os.path.join(path, target_file)
    print(f"File yang akan digunakan: {csv_path}")
else:
    csv_files = [f for f in files if f.endswith('.csv')]
    csv_path = os.path.join(path, csv_files[0]) if csv_files else None
    print(f"File target tidak langsung ketemu, fallback ke: {csv_path}")

Using Colab cache for faster access to the 'spotify-songs-with-attributes-and-lyrics' dataset.
Path ke folder dataset: /kaggle/input/spotify-songs-with-attributes-and-lyrics
File yang akan digunakan: /kaggle/input/spotify-songs-with-attributes-and-lyrics/songs_with_attributes_and_lyrics.csv


## Langkah 2 - Boilerplate

In [7]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Load dataset
# -------------------------------
# Menggunakan csv_path hasil dari download kagglehub
try:
    df = pd.read_csv(csv_path)
except NameError:
    # Fallback jika cell kagglehub belum dijalankan
    df = pd.read_csv('spotify_songs.csv')

features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

## Langkah 3 - Cek Dataset

In [8]:
print("Ukuran dataset:", df.shape)
print("Fitur yang digunakan:", features)
display(df.head())


Ukuran dataset: (955320, 17)
Fitur yang digunakan: ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']


,id,name,album_name,artists,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,lyrics
0,0Prct5TDjAnEgIqbxcldY9,!,UNDEN!ABLE,HELLYEAH,0.415,0.6050,7,-11.157,1,0.0575,0.00116,0.838000,0.4710,0.193,100.059,79500.0,"He said he came from Jamaica,\n he owned a cou..."
1,2ASl4wirkeYm3OWZxXKYuq,!!,NaN,Yxngxr1,0.788,0.6480,7,-9.135,0,0.3150,0.90000,0.000000,0.1760,0.287,79.998,114000.0,"Fucked a bitch, now she running with my kids\n..."
2,69lcggVPmOr9cvPx9kLiiN,!!! - Interlude,Where I Belong EP,Glowie,0.000,0.0354,7,-20.151,0,0.0000,0.90800,0.000000,0.4790,0.000,0.000,11413.0,"Oh, my God, I'm going crazy\n"
3,4U7dlZjg1s9pjdppqZy0fm,!!De Repente!!,Un Palo Al Agua (20 Grandes Canciones),Rosendo,0.657,0.8820,5,-6.340,1,0.0385,0.00740,0.000013,0.0474,0.939,123.588,198173.0,Continuamente se extraña la gente si no puede ...
4,4v1IBp3Y3rpkWmWzIlkYju,!!De Repente!!,Fuera De Lugar,Rosendo,0.659,0.8930,5,-8.531,1,0.0411,0.09220,0.000019,0.0534,0.951,123.600,199827.0,Continuamente se extraña la gente si no puede ...


## Langkah 4 - Exact NN


In [9]:
# Pastikan tipe data float32 agar kompatibel dengan FAISS dan HNSW.
X_scaled = np.asarray(X_scaled, dtype='float32')

query_index = 0
query = X_scaled[query_index:query_index + 1]

nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)

start = time.time()
dist_exact, idx_exact = nn.kneighbors(query)
time_exact = time.time() - start

print("Exact NN indices:", idx_exact[0])
print("Waktu Exact:", time_exact, "s")


Exact NN indices: [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]
Waktu Exact: 0.053194522857666016 s


## Langkah 5 - ANNOY

In [10]:
f = X_scaled.shape[1]
annoy_index = AnnoyIndex(f, 'euclidean')

start = time.time()
for i, v in enumerate(X_scaled):
    annoy_index.add_item(i, v)
annoy_index.build(10)
build_annoy = time.time() - start

start = time.time()
idx_annoy = annoy_index.get_nns_by_vector(query[0], k)
time_annoy = time.time() - start

print("ANNOY indices:", idx_annoy)
print("Build ANNOY:", build_annoy, "s")
print("Query ANNOY:", time_annoy, "s")


ANNOY indices: [0, 764272, 749223, 833164, 676733, 523698, 85956, 763625, 906004, 385544]
Build ANNOY: 9.223400592803955 s
Query ANNOY: 0.00020170211791992188 s


## Langkah 6 - FAISS

In [11]:
d = X_scaled.shape[1]
nlist = 100
quantizer = faiss.IndexFlatL2(d)
faiss_index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)

start = time.time()
faiss_index.train(X_scaled)
faiss_index.add(X_scaled)
build_faiss = time.time() - start

faiss_index.nprobe = 10

start = time.time()
dist_faiss, idx_faiss = faiss_index.search(query, k)
time_faiss = time.time() - start

print("FAISS indices:", idx_faiss[0])
print("Build FAISS:", build_faiss, "s")
print("Query FAISS:", time_faiss, "s")


FAISS indices: [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]
Build FAISS: 0.1764664649963379 s
Query FAISS: 0.008951187133789062 s


## Langkah 7 - HNSW

In [12]:
hnsw_index = hnswlib.Index(space='l2', dim=d)

start = time.time()
hnsw_index.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
hnsw_index.add_items(X_scaled)
build_hnsw = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
idx_hnsw, dist_hnsw = hnsw_index.knn_query(query, k=k)
time_hnsw = time.time() - start

print("HNSW indices:", idx_hnsw[0])
print("Build HNSW:", build_hnsw, "s")
print("Query HNSW:", time_hnsw, "s")


HNSW indices: [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]
Build HNSW: 118.49101138114929 s
Query HNSW: 0.0003094673156738281 s


## Langkah 8 - Perbandingan Hasil

In [13]:
def recall_at_k(pred, gt, k):
    return len(set(pred[:k]).intersection(gt[:k])) / k


hasil = pd.DataFrame([
    {
        'Algoritma': 'Exact NN',
        'Build (s)': 0.0,
        'Query (s)': time_exact,
        'Recall@10': 1.0,
    },
    {
        'Algoritma': 'ANNOY',
        'Build (s)': build_annoy,
        'Query (s)': time_annoy,
        'Recall@10': recall_at_k(idx_annoy, idx_exact[0], k),
    },
    {
        'Algoritma': 'FAISS',
        'Build (s)': build_faiss,
        'Query (s)': time_faiss,
        'Recall@10': recall_at_k(idx_faiss[0], idx_exact[0], k),
    },
    {
        'Algoritma': 'HNSW',
        'Build (s)': build_hnsw,
        'Query (s)': time_hnsw,
        'Recall@10': recall_at_k(idx_hnsw[0], idx_exact[0], k),
    },
])

display(hasil)


,Algoritma,Build (s),Query (s),Recall@10
0,Exact NN,0.000000,0.053195,1.0
1,ANNOY,9.223401,0.000202,0.6
2,FAISS,0.176466,0.008951,1.0
3,HNSW,118.491011,0.000309,1.0


## Langkah 9 - Tampilkan Track Terdekat



In [14]:
info_cols = [
    c for c in ['track_name', 'track_artist', 'artist_name', 'track_album_name']
    if c in df.columns
]

print("Query track:")
display(df.iloc[[query_index]][info_cols] if info_cols else df.iloc[[query_index]].head())

print("Track terdekat menurut ANNOY:")
display(df.iloc[idx_annoy][info_cols] if info_cols else df.iloc[idx_annoy].head())


Query track:


,id,name,album_name,artists,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,lyrics
0,0Prct5TDjAnEgIqbxcldY9,!,UNDEN!ABLE,HELLYEAH,0.415,0.605,7,-11.157,1,0.0575,0.00116,0.838,0.471,0.193,100.059,79500.0,"He said he came from Jamaica,\n he owned a cou..."


Track terdekat menurut ANNOY:


,id,name,album_name,artists,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,lyrics
0,0Prct5TDjAnEgIqbxcldY9,!,UNDEN!ABLE,HELLYEAH,0.415,0.605,7,-11.157,1,0.0575,0.001160,0.838,0.471,0.193,100.059,79500.0,"He said he came from Jamaica,\n he owned a cou..."
764272,4ikbMYjg2Yn2zjB4TeiX8P,Testimonium,Testimonium,Lacrimosa,0.430,0.573,9,-8.996,0,0.0309,0.000008,0.768,0.367,0.188,101.990,603520.0,Nicht geb' ich ab mein Dasein\n Wie ein Sterbe...
749223,2KvIO4ew1TGnz1VfTJk49g,Sæglópur,NaN,Sigur Rós,0.351,0.609,9.0,-9.031,1.0,0.0418,0.061400,0.803,0.386,0.100,102.041,504296.0,"Þú, ert refur\n Þú, þú hefur - á lofti\n Á lof..."
833164,0pcLVqykiOhWTSOcmYJKIb,Turn from Heaven,NaN,Therion,0.484,0.645,5,-10.189,1,0.0280,0.000212,0.912,0.606,0.173,94.967,186933.0,"Morning\n Birth of the sun (Morning, Morning, ..."
676733,1S8jdYiBiX2b7awbS0LuRo,See You When I Die,NaN,BOYO,0.394,0.492,2,-11.229,1,0.0263,0.162000,0.855,0.372,0.115,92.487,252973.0,I'll see you when I die\n I'll see you when I\...
